# MCDA Scoring — Site Suitability Recommender

Implement AHP pairwise weighting, Weighted Linear Combination (WLC),
and composite scoring for site suitability ranking.

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

## MCDA Steps

1. Define AHP pairwise comparison matrix
2. Compute priority weights via eigenvector method
3. Check consistency ratio (CR < 0.10)
4. Apply Weighted Linear Combination (WLC)
5. Rank candidate sites by composite score

In [ ]:
# AHP Pairwise Comparison Matrix
criteria = ['soil_quality', 'elevation', 'slope', 'ndvi', 'water_proximity', 'road_proximity']
n = len(criteria)

# Pairwise comparison (Saaty scale 1-9)
ahp_matrix = np.array([
    [1,   3,   5,   2,   3,   4],   # soil_quality
    [1/3, 1,   3,   1/2, 1,   2],   # elevation
    [1/5, 1/3, 1,   1/3, 1/2, 1],   # slope
    [1/2, 2,   3,   1,   2,   3],   # ndvi
    [1/3, 1,   2,   1/2, 1,   2],   # water_proximity
    [1/4, 1/2, 1,   1/3, 1/2, 1],   # road_proximity
])

# Eigenvector method for weights
eigenvalues, eigenvectors = np.linalg.eig(ahp_matrix)
max_idx = np.argmax(eigenvalues.real)
weights = eigenvectors[:, max_idx].real
weights = weights / weights.sum()

# Consistency check
lambda_max = eigenvalues[max_idx].real
CI = (lambda_max - n) / (n - 1)
RI = {3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24, 7: 1.32}[n]
CR = CI / RI

print('AHP Weights:')
for c, w in zip(criteria, weights):
    print(f'  {c:20s}: {w:.4f}')
print(f'\nConsistency Ratio: {CR:.4f} ({"PASS" if CR < 0.10 else "FAIL"})')

In [ ]:
# Weighted Linear Combination
sites = gpd.read_file('../data/processed/sites_normalized.shp')

sites['suitability_score'] = sum(
    weights[i] * sites[criteria[i]] for i in range(n)
)

# Rank sites
sites['rank'] = sites['suitability_score'].rank(ascending=False).astype(int)
sites = sites.sort_values('rank')

print('Top 10 sites:')
print(sites[['site_id', 'suitability_score', 'rank']].head(10).to_string(index=False))

sites.to_file('../data/processed/sites_ranked.shp')